# 13 — Universal Construction (Threshold Sweep)

**Purpose:** Apply 9 combinations of epsilon x consistency to raw activations from notebook 12. Build binary masks for both objects (with marginalization to universals) and checkers. No model needed — just numpy thresholding.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np, pandas as pd, h5py
from tqdm.auto import tqdm

EPSILONS = [0.001, 0.1, 0.5]
CONSISTENCIES = [0.2, 0.5, 0.8]
N_LAYERS = 8

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

OBJECT_ACTIVATIONS = "12_object_activations.h5"
CHECKER_ACTIVATIONS = "12_checker_activations.h5"

print(f"Epsilons: {EPSILONS}")
print(f"Consistencies: {CONSISTENCIES}")
print(f"Combinations: {len(EPSILONS) * len(CONSISTENCIES)}")

In [ ]:
# Cell 2 – Load raw activations
from module2.io_utils import load_activations_hdf5

obj_data = load_activations_hdf5(os.path.join(DATA_DIR, OBJECT_ACTIVATIONS))
obj_activations = obj_data["pair_activations"]
print(f"Object activations: {len(obj_activations)} pairs")

chk_data = load_activations_hdf5(os.path.join(DATA_DIR, CHECKER_ACTIVATIONS))
chk_activations = chk_data["pair_activations"]
print(f"Checker activations: {len(chk_activations)} entries")

In [ ]:
# Cell 3 – Threshold function
def apply_thresholds(pair_activations, epsilon, consistency):
    """Apply epsilon + consistency thresholds to raw activations. Returns {key: {layer: bool_mask}}."""
    pair_masks = {}
    for key, raw in pair_activations.items():
        n_prompts = raw["n_prompts"]
        masks = {}
        for lid, layer_data in raw["layers"].items():
            mean_act = layer_data["activation_sum"] / n_prompts
            firing_rate = layer_data["firing_count"].astype(np.float32) / n_prompts
            mask = (mean_act > epsilon) & (firing_rate >= consistency)
            masks[lid] = mask
        pair_masks[key] = masks
    return pair_masks

print("Threshold function defined.")

In [ ]:
# Cell 4 – Object sweep: apply thresholds + compute universals
from module2.marginalization import UniversalModuleComputer
from module2.io_utils import save_atlas_hdf5
from module2.metrics import compute_jaccard_matrix

computer = UniversalModuleComputer()
ast_nodes = sorted(set(a for a, _ in obj_activations))
builtin_objs = sorted(set(b for _, b in obj_activations))

obj_summary = []

for eps in EPSILONS:
    for cons in CONSISTENCIES:
        print(f"\n--- Object: eps={eps}, cons={cons} ---")
        pair_masks = apply_thresholds(obj_activations, eps, cons)
        
        # Compute universals
        universal_masks = computer.compute_all(pair_masks, ast_nodes, builtin_objs)
        
        # Jaccard matrix at layer 4
        metrics = {}
        rep_layer = 4
        if universal_masks.get("ast"):
            metrics["jaccard_ast_matrix"] = compute_jaccard_matrix(universal_masks["ast"], rep_layer)
            metrics["ast_names"] = sorted(universal_masks["ast"].keys())
        if universal_masks.get("builtin"):
            metrics["jaccard_builtin_matrix"] = compute_jaccard_matrix(universal_masks["builtin"], rep_layer)
            metrics["builtin_names"] = sorted(universal_masks["builtin"].keys())
        
        # Save
        out_name = f"13_object_masks_eps{eps}_cons{cons}.h5"
        out_path = os.path.join(DATA_DIR, out_name)
        meta = {
            "epsilon": eps, "consistency_thresh": cons,
            "n_pairs": len(pair_masks), "n_layers": N_LAYERS,
            "ast_nodes": ast_nodes, "builtin_objs": builtin_objs,
        }
        save_atlas_hdf5(out_path, pair_masks, universal_masks, metrics, meta)
        
        n_ast = len(universal_masks.get("ast", {}))
        n_blt = len(universal_masks.get("builtin", {}))
        # Mean circuit size at layer 4
        sizes = []
        for lm in pair_masks.values():
            if 4 in lm:
                sizes.append(lm[4].sum())
        mean_size = np.mean(sizes) if sizes else 0
        
        print(f"  Universal AST: {n_ast}, Builtin: {n_blt}, Mean pair size L4: {mean_size:.1f}")
        print(f"  Saved: {out_name}")
        
        obj_summary.append({
            "epsilon": eps, "consistency": cons, "type": "object",
            "n_universal_ast": n_ast, "n_universal_builtin": n_blt,
            "mean_pair_size_L4": round(mean_size, 1),
        })

In [ ]:
# Cell 5 – Checker sweep: apply thresholds (no marginalization needed)

chk_summary = []

for eps in EPSILONS:
    for cons in CONSISTENCIES:
        print(f"\n--- Checker: eps={eps}, cons={cons} ---")
        checker_masks = apply_thresholds(chk_activations, eps, cons)
        
        # Save using h5py directly (checker has no pair/universal structure)
        out_name = f"13_checker_masks_eps{eps}_cons{cons}.h5"
        out_path = os.path.join(DATA_DIR, out_name)
        
        with h5py.File(out_path, "w") as f:
            tcm = f.create_group("token_checker_masks")
            for (obj_type, obj_name), layers in checker_masks.items():
                obj_key = f"{obj_type}__{obj_name}"
                for lid, mask in layers.items():
                    tcm.create_dataset(
                        f"layer_{lid}/{obj_key}",
                        data=mask.astype(np.bool_),
                        compression="gzip"
                    )
            md = f.create_group("metadata")
            md.attrs["epsilon"] = eps
            md.attrs["consistency_threshold"] = cons
        
        # Stats
        sizes = []
        for layers in checker_masks.values():
            if 4 in layers:
                sizes.append(layers[4].sum())
        mean_size = np.mean(sizes) if sizes else 0
        
        print(f"  Checker masks: {len(checker_masks)}, Mean size L4: {mean_size:.1f}")
        print(f"  Saved: {out_name}")
        
        chk_summary.append({
            "epsilon": eps, "consistency": cons, "type": "checker",
            "n_masks": len(checker_masks), "mean_size_L4": round(mean_size, 1),
        })

In [ ]:
# Cell 6 – Summary table
obj_df = pd.DataFrame(obj_summary)
chk_df = pd.DataFrame(chk_summary)

print("=== OBJECT UNIVERSAL CIRCUITS ===")
display(obj_df)
print("\n=== CHECKER MASKS ===")
display(chk_df)

# Flag empty results
for _, row in obj_df.iterrows():
    if row["n_universal_ast"] == 0 and row["n_universal_builtin"] == 0:
        print(f"WARNING: No universals at eps={row['epsilon']}, cons={row['consistency']}")

In [ ]:
# Cell 7 – Done
print("13 complete.")
print(f"Produced {len(EPSILONS) * len(CONSISTENCIES)} object mask files + {len(EPSILONS) * len(CONSISTENCIES)} checker mask files")
print(f"Total: {2 * len(EPSILONS) * len(CONSISTENCIES)} HDF5 files in {DATA_DIR}")